# Extract Central Bankers From Wikipedia Categories

Este notebook usa un camino distinto al de `List of central banks` + página de cada banco central. Aquí partimos desde la categoría de Wikipedia `Category:Central_bankers`.

## Qué hace este flujo

1. Descarga la página `List of central banks` para construir una tabla base `central_banks_df` con:
   - `country`
   - `central_bank`
   - `wikipedia_bank_url`

2. Entra a `Category:Central_bankers` en Wikipedia y busca sus subcategorías.

3. Filtra solo las subcategorías que parecen útiles para este proyecto, por ejemplo:
   - `Governors of ...`
   - `Presidents of ...`
   - `Chairs of ...`

4. Entra a cada subcategoría y extrae los nombres de personas que aparecen en esa categoría.

5. Guarda un dataset intermedio `central_bankers_categories_df` donde cada fila sigue siendo una categoría, pero con una lista de nombres en la columna `governor_names`.

6. Expande esa lista para construir una tabla final en formato long: una fila por persona.

7. A partir del nombre de la categoría infiere:
   - la posición (`Governor`, `President`, `Chair`)
   - el nombre del banco central

8. Luego cruza el nombre del banco central contra `central_banks_df` para recuperar el país.

9. Finalmente exporta el archivo `governors_clean_names.csv`.

## Qué contiene cada tabla importante

### `central_banks_df`
Lista base de bancos centrales obtenida desde `List of central banks`.

### `central_bankers_categories_df`
Una fila por subcategoría de Wikipedia, con:
- `category_name`
- `category_url`
- `governor_names` como lista de nombres
- `governor_count`

### `df`
Tabla final limpia con una fila por persona, incluyendo:
- `country`
- `central_bank_name`
- `PName_original`
- `PName`
- `first`
- `last`
- `Position`
- `category_name`
- `category_url`

## Idea importante

Este enfoque usa las categorías de Wikipedia como fuente de nombres. Eso lo hace útil para construir una base rápida de personas por banco central, pero depende de cómo Wikipedia haya nombrado y organizado esas categorías.

In [ ]:
import re
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; CentralBankNotebook/1.0)"}
WIKI_BANKS_URL = "https://en.wikipedia.org/wiki/List_of_central_banks"
CATEGORY_URL = "https://en.wikipedia.org/wiki/Category:Central_bankers"


def clean_text(value):
    value = str(value or "")
    value = re.sub(r"\[[^\]]*\]", "", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


# 1. Build central_banks_df from List of central banks
banks_response = requests.get(WIKI_BANKS_URL, headers=HEADERS, timeout=30)
banks_response.raise_for_status()

banks_soup = BeautifulSoup(banks_response.text, "html.parser")
html_tables = banks_soup.select("table.wikitable")
wiki_tables = pd.read_html(StringIO(banks_response.text))

if len(html_tables) < 1 or len(wiki_tables) < 1:
    raise ValueError("Could not find the main central bank table on Wikipedia")

alphabetical_html_table = html_tables[0]
alphabetical_records = []

for row in alphabetical_html_table.select("tr")[1:]:
    cols = row.find_all(["th", "td"])
    if len(cols) < 3:
        continue

    country = clean_text(cols[0].get_text(" ", strip=True))
    bank_col = cols[2]
    central_bank = clean_text(bank_col.get_text(" ", strip=True))
    link = bank_col.find("a", href=True)
    wikipedia_bank_url = ""
    if link and link["href"].startswith("/wiki/"):
        wikipedia_bank_url = "https://en.wikipedia.org" + link["href"]

    if country and central_bank:
        alphabetical_records.append(
            {
                "country": country,
                "central_bank": central_bank,
                "wikipedia_bank_url": wikipedia_bank_url,
            }
        )

alphabetical_banks_df = pd.DataFrame(alphabetical_records)

if len(wiki_tables) > 1:
    major_table_raw = wiki_tables[1].copy()
    major_banks_df = major_table_raw.rename(
        columns={
            major_table_raw.columns[0]: "country",
            major_table_raw.columns[1]: "central_bank",
        }
    )[["country", "central_bank"]].copy()
    major_banks_df["wikipedia_bank_url"] = ""
else:
    major_banks_df = pd.DataFrame(columns=["country", "central_bank", "wikipedia_bank_url"])

for frame in (alphabetical_banks_df, major_banks_df):
    if frame.empty:
        continue
    frame["country"] = frame["country"].astype(str).str.strip()
    frame["central_bank"] = frame["central_bank"].astype(str).str.strip()
    frame["wikipedia_bank_url"] = frame["wikipedia_bank_url"].astype(str).str.strip()

central_banks_df = pd.concat([alphabetical_banks_df, major_banks_df], ignore_index=True)
central_banks_df = central_banks_df.drop_duplicates(subset=["country", "central_bank"]).reset_index(drop=True)

print("central_banks_df:", central_banks_df.shape)
display(central_banks_df.head(10))


# 2. Extract relevant subcategories from Category:Central_bankers
category_response = requests.get(CATEGORY_URL, headers=HEADERS, timeout=30)
category_response.raise_for_status()

category_soup = BeautifulSoup(category_response.text, "html.parser")
category_section = category_soup.select_one("#mw-subcategories")

category_rows = []
if category_section:
    for link in category_section.select("a[href]"):
        href = link.get("href", "")
        if not href.startswith("/wiki/Category:"):
            continue

        category_rows.append(
            {
                "category_name": clean_text(link.get_text(" ", strip=True)),
                "category_url": "https://en.wikipedia.org" + href,
            }
        )

central_bankers_categories_df = pd.DataFrame(category_rows).drop_duplicates()
central_bankers_categories_df = central_bankers_categories_df[
    central_bankers_categories_df["category_name"].str.contains(
        r"governors?|presidents?|chair(men|women|persons?)?",
        case=False,
        na=False,
    )
].reset_index(drop=True)

print("central_bankers_categories_df:", central_bankers_categories_df.shape)
display(central_bankers_categories_df.head(20))


# 3. Extract the names inside each category
def extract_people_from_category(category_url):
    response = requests.get(category_url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    pages_section = soup.select_one("#mw-pages")

    names = []
    if pages_section:
        for item in pages_section.select("div.mw-category-group li"):
            link = item.find("a", href=True)
            if not link:
                continue

            href = link.get("href", "")
            if not href.startswith("/wiki/") or href.startswith("/wiki/Category:"):
                continue

            names.append(clean_text(link.get_text(" ", strip=True)))

    return sorted(set(name for name in names if name))


central_bankers_categories_df = central_bankers_categories_df.copy()
central_bankers_categories_df["governor_names"] = central_bankers_categories_df["category_url"].apply(extract_people_from_category)
central_bankers_categories_df["governor_count"] = central_bankers_categories_df["governor_names"].apply(len)

display(central_bankers_categories_df.head(10))


# 4. Expand to one row per person and recover the country
df = central_bankers_categories_df.copy()
df = df.explode("governor_names").reset_index(drop=True)
df = df.rename(columns={"governor_names": "PName_original"})
df = df[df["PName_original"].notna()].copy()

df["PName_original"] = df["PName_original"].astype(str).str.strip()
df = df[df["PName_original"] != ""]
df = df[df["PName_original"].str.split().str.len() >= 2]


def infer_position(category_name):
    text = str(category_name).lower()
    if "governor" in text:
        return "Governor"
    if "president" in text:
        return "President"
    if "chair" in text:
        return "Chair"
    return ""


def infer_bank_name(category_name):
    text = str(category_name)
    prefixes = [
        "Governors of ",
        "Presidents of ",
        "Chairmen of ",
        "Chairwomen of ",
        "Chairpersons of ",
        "Chairs of ",
    ]
    for prefix in prefixes:
        if text.startswith(prefix):
            return text.replace(prefix, "").strip()
    return text.strip()


bank_lookup_df = central_banks_df[["country", "central_bank"]].drop_duplicates().copy()
bank_lookup_df["central_bank_norm"] = bank_lookup_df["central_bank"].str.lower().str.strip()
bank_to_country = dict(zip(bank_lookup_df["central_bank_norm"], bank_lookup_df["country"]))

df["central_bank_name"] = df["category_name"].apply(infer_bank_name)
df["country"] = df["central_bank_name"].str.lower().str.strip().map(bank_to_country).fillna("")
df["PName"] = df["PName_original"]
df[["first", "last"]] = df["PName"].str.split(" ", n=1, expand=True)
df["last"] = df["last"].fillna("")
df["Position"] = df["category_name"].apply(infer_position)

df = df.drop_duplicates(subset=["country", "central_bank_name", "PName_original", "Position"])
df = df[["country", "central_bank_name", "PName_original", "PName", "first", "last", "Position", "category_name", "category_url"]]
df = df.sort_values(["country", "central_bank_name", "PName_original"]).reset_index(drop=True)

df.to_csv("governors_clean_names.csv", index=False, sep=";")
display(df.head(20))
print("\nTotal names:", len(df))
